# PC 9: Numerical solution of parabolic PDEs


In [ ]:
# NumPy provides efficient array operations.
import numpy as np

# SciPy sparse matrices.
import scipy.sparse as sparse
# Sparse LU factorization and sparse linear solver.
from scipy.sparse.linalg import splu, spsolve
# Euclidean norm.
from numpy.linalg import norm

# Matplotlib is used for plotting.
from matplotlib import pyplot as plt

## Exercise 1: Heat equation

>Let $\nu>0$ be a diffusion coefficient, $\Omega=]0,L[$ an interval with $L>0$, and $u^0\in C^0(\Omega)$ an initial condition. Consider the heat equation
>
>$$ \left\{ \begin{aligned}
      &\displaystyle\frac{\partial u}{\partial t}(t,x) - \nu\frac{\partial^2 u}{\partial x^2}(t,x) = 0, \quad &\forall (t,x) \in\mathbb{R}_+^*\times\Omega, \\ 
      &\frac{\partial u}{\partial x}(t,0) = 0 = \frac{\partial u}{\partial x}(t,L), \quad &\forall t\in\mathbb{R}_+^*, \\ 
      &u(0,x) = u^0(x), \quad &\forall x\in\Omega.
    \end{aligned}
    \right. \qquad{} (1) $$
>
>  We admit the following results about the PDE (1), and we will try to prove their discrete counterparts:
>- The heat equation (1) has a unique solution.
>- **Maximum principle:**
>
>$$  \forall~0\leq s<t<\infty,\ \forall x\in\Omega, \quad{} \min_{y\in\Omega} u(s,y) \leq u(t,x) \leq \max_{y\in\Omega} u(s,y).  $$
>
>- **Mass conservation:**
>
>  $$ \forall~0\leq t<\infty, \quad{} \int_\Omega u(t,x)\, dx = \int_\Omega u^0(x)\, dx.  $$
>
>- **Energy decay:**
>
>  $$ \forall~0\leq s<t<\infty, \quad{} \int_\Omega \vert u(t,x)\vert^2 dx \leq \int_\Omega \vert u(s,x)\vert^2 dx. $$
>
> To study the heat equation (1) numerically, we consider several numerical methods and study their properties.
>- For the time discretization, we fix a final time $T>0$ and consider
>
> $$ N\in\mathbb{N}^*, \qquad{} \Delta t = \frac{T}{N}, \quad{} \text{and} \quad{} t^n = n\Delta t,\ n = 0, \dots, N. $$
>
>- For the space discretization, we consider
>
> $$ J\in\mathbb{N}^*, \qquad{} \Delta x = \frac{L}{J}, \quad{} \text{and}\quad{} x_j = j\Delta x,\ j = -1, \dots, J+1. $$
>
>- We write
>
> $$ U^0_j = u^0(x_j),\quad{} j = 0, \dots, J, $$
>
>  and we iteratively construct a sequence $U^n = \left(U^n_j\right)_{0\leq j \leq J}$ such that $U^n_j$ approximates $u(t^n,x_j)$.
>
>For the implementation questions of this exercise, we take
>
> $$  \nu = 1,\quad{} L=10,\quad{} J=100, \quad{} u^0(x) = 2 - \cos \frac{2\pi x}{L},  $$
>
>and different values of $T$ and $\Delta t$ given below.


### Question 1: Exact solution

Check that the solution of (1) with the initial condition above is
$$ u(t,x) = 2 - e^{-\nu\frac{4\pi^2}{L^2}t}\cos\left(\frac{2\pi x}{L}\right). $$


### Question 2: Explicit finite difference scheme

>Consider the scheme
>
> $$ \frac{U^{n+1}_j - U^n_j}{\Delta t} - \nu \frac{U^n_{j-1}-2U^n_j+U^n_{j+1}}{(\Delta x)^2} = 0 \qquad{} \forall~n\geq 0,\quad{} \forall~0 \leq j\leq J.\qquad{} (2) $$
>
>For $j=0$ (resp. $j=J$), this scheme involves $U^n_{-1}$ (resp. $U^n_{J+1}$). These two quantities are determined by the boundary conditions. Discretizing the Neumann conditions with one-sided or centered differences gives
>
> $$ \frac{U_0^n - U^n_{-1}}{\Delta x} = 0 \quad{}\text{and}\quad{} \frac{U^n_{J+1} - U^n_{J}}{\Delta x} = 0, \qquad{}(3a) $$
>
>or
>
> $$ \frac{U_1^n - U^n_{-1}}{2\Delta x} = 0 \quad{}\text{and}\quad{} \frac{U^n_{J+1} - U^n_{J-1}}{2\Delta x} = 0. \qquad{} (3b) $$
>
>The advantages and drawbacks of these two discretizations will be studied throughout the questions.


1) **Matrix form of the scheme**

Give the $(J+1)\times(J+1)$ matrices $A^{expl}_1$ and $A^{expl}_2$ such that scheme (2) reads $U^{n+1}=A^{expl}_1 U^n$ with convention (3a), and $U^{n+1}=A^{expl}_2 U^n$ with convention (3b).

*Hint: introduce the dimensionless number $\alpha = \frac{\nu \Delta t}{\Delta x^2}$.*


2) **Numerical properties**

a) **Consistency**

>We define the local truncation error $\varepsilon^n=\left(\varepsilon_j^n\right)_{0\leq j\leq J}$ of scheme (2) by
>
> $$ \varepsilon_j^n = \frac{u(t^{n+1},x_j) - u(t^n,x_j)}{\Delta t} - \nu \frac{u(t^n,x_{j-1}) - 2u(t^n,x_j) + u(t^n,x_{j+1})}{(\Delta x)^2}, $$
>
>for all $1\le j \le J-1$, and
>
>$$\begin{aligned}
  \varepsilon_0^n   &= \frac{u(t^{n+1},x_0   ) - u(t^n,x_0   )}{\Delta t} - \nu \frac{-2u(t^n,x_0)+2u(t^n,x_{1})}{(\Delta x)^2}, \\
  \varepsilon_{J}^n &= \frac{u(t^{n+1},x_{J}) - u(t^n,x_{J})}{\Delta t} - \nu \frac{2u(t^n,x_{J-1})-2u(t^n,x_{J})}{(\Delta x)^2},
\end{aligned}$$ 
>  
>when using the boundary conditions (3b).
>The scheme is consistent if $\lim\limits_{\Delta t, \Delta x \rightarrow 0} \max\limits_{j,n} |\varepsilon_j^n| = 0$, and of order $p$ in time and $q$ in space if there exists $C\in\mathbb{R}_+$ such that $\max\limits_{j,n} |\varepsilon_j^n| \le C (\Delta t^p + \Delta x^q)$.

Show that scheme (2) with the boundary conditions (3b) is consistent, and give its order. *You may assume that the solution is as smooth as needed.*

b) **Stability in the $\ell_{\Delta x}^2$ norm**

>Scheme (2) with the boundary conditions (3a) or (3b) is stable for the norm $\| \cdot \|$ if there exists a constant $C=C(T)\in\mathbb{R}^+$, independent of $\Delta x$ and $\Delta t$ but possibly depending on $T$, such that, for every initial condition $U^0$,
>
> $$ \|U^n\| \le C(T) \|U^0\| \quad{}\forall~n\ge 0. $$
>
>We define the $\ell^2_{\Delta x}$ norm of $U^n$ by
>
> $$\begin{aligned} \left\Vert U^n \right\Vert_{\ell^2_{\Delta x}} := \sqrt{\Delta x \sum_{j=0}^{J} \left(U^n_j\right)^2} = \sqrt{\Delta x} \left\Vert U^n \right\Vert_{2}.\end{aligned}$$
>
>**Remark:** if $U^n_j$ approximates $u(t^n,x_j)$ well, this norm approximates the $L^2$ norm of $x\mapsto u(t^n,x)$ as $\Delta x\to 0$.

Give a sufficient condition on $\alpha$ for scheme (2-3a) to be stable in the $\ell^2_{\Delta x}$ norm.

*Hint: study the spectrum of $A_1^{expl}$ using the Gershgorin theorem.*

In what follows, we assume that scheme (2-3b) is also stable for this norm under the same condition.

c) **Convergence in the $\ell_{\Delta x}^2$ norm**

>A scheme is convergent in the norm $\| \cdot \|$ if
>
>$$ \lim\limits_{\Delta t, \Delta x \rightarrow 0}\sup\limits_{t^n \le T} \|e^n\| = 0, $$
>
>where $e^n$ is the error vector given by $e_j^n = U_j^n - u(t^n,x_j)$, $j=0,\ldots,J$.

Show that scheme (2-3b) converges in the $\ell_{\Delta x}^2$ norm under the stability condition found in the previous question.

*Hint: as for ODEs, write $\left(u(t^{n+1},x_j)\right)_{0\leq j \leq J}$ as a function of $\left(u(t^n,x_j)\right)_{0\leq j \leq J}$, $A^{expl}$, and $\varepsilon^n$.*


3) **Physical properties**

a) **Discrete maximum principle:** Show that, under the stability condition on $\alpha$ found in 2.b, scheme (2) satisfies
$$\begin{aligned} \forall~n\geq 0,\ \forall~0\leq j\leq J, \quad{} \min_i U^n_i \leq U^{n+1}_j \leq \max_i U^n_i, \end{aligned}$$  
for both choices (3a) and (3b) of boundary discretization.

>**Remark:** this shows that the scheme is also stable in the infinity norm.

b) **Mass conservation:** We show that both schemes conserve an approximation of the mass $\int_\Omega u(t,x)\, dx$. The approximate mass at step $n$ is
$$\Delta x \sum_{j=0}^{J} U^{n}_j \qquad{} \text{for the boundary conditions (3a)},$$
and
$$\Delta x \left(\frac{1}{2}U^{n}_0 +  \sum_{j=1}^{J-1} U^{n}_j + \frac{1}{2}U^{n}_J\right) \qquad{} \text{for the boundary conditions (3b)}.$$
The integral is approximated with a rectangle rule in the first case (with one extra rectangle, to keep the symmetry of the problem) and with the trapezoidal rule in the second case. In both cases, show that the approximate mass is conserved by the scheme.

*Hint: in each case, write this quantity as $\Delta x\, v^T U^n$ for a suitable vector $v\in\mathbb{R}^{J+1}$.*

>**Remark: energy decay.** The $\ell_{\Delta x}^2$ stability condition of the previous question gives $\|U^{n+1}\|_{\ell_{\Delta x}^2} \le \|U^n\|_{\ell_{\Delta x}^2}$, which is the discrete version of the energy decay of the continuous solution.


4) **Implementation:** Recall the numerical parameters
$$  \Delta x = \frac{L}{J}, \quad{} N \Delta t = T, \quad{} \alpha = \frac{\nu \Delta t}{\Delta x^2}, $$
and set
$$ \nu = 1, \quad{} L=10,\quad{} J=100, \quad{} u^0(x) = 2 - \cos \frac{2\pi x}{L}.  $$

a) Implement schemes (2-3a) and (2-3b).

**Start by building the two matrices $A^{expl}$ computed in 1) in sparse format, using the function `sparse.diags` of SciPy. Then implement a single function that takes the matrix of the scheme as input and applies the scheme, using `A.dot` for matrix-vector products (it is suited to the sparse format).**

b) For each scheme, use $\Delta t= 0.004$ and $T=5$, and plot the final numerical solution, together with the initial condition and the exact solution at the final time.

c) In each case, check that the corresponding approximate mass is the same at the initial and final times.

d) For each scheme, then use $\Delta t= 0.0051$. Explain the differences in behavior compared with $\Delta t= 0.004$.


In [ ]:
def A_explicite1(alpha, J):
    """
    Compute the matrix A_expl of scheme (2-3a) in sparse format.
    """
    # Your code here.
    pass

def A_explicite2(alpha, J):
    """
    Compute the matrix A_expl of scheme (2-3b) in sparse format.
    """
    # Your code here.
    pass


In [ ]:
# Test cell to check the entries of these matrices.
alpha = 0.1
J = 4
print("A1 = \n", A_explicite1(alpha, J).toarray(), "\n")
print("A2 = \n", A_explicite2(alpha, J).toarray())

In [ ]:
def schema_explicite_chaleur(U0, A, N, dx):
    """
    Compute the iterates U^n of scheme (2-3a) or (2-3b), and return U^N.
    """
    # Your code here.
    pass


In [ ]:
def u0_1(x):
    # Initial condition of Exercise 1.
    return 2 - np.cos(2*np.pi*x/L)

def u_th(t, x):
    # Exact solution of Exercise 1.
    return 2 - np.exp(-nu*4*np.pi**2/L**2*t)*np.cos(2*np.pi*x/L)

In [ ]:
# Questions 4.b to 4.d: numerical experiments with dt = 0.004 and dt = 0.0051.

# Your code here.


### Question 3: Implicit finite difference scheme

>Consider the scheme
>
>$$ \frac{U^{n+1}_j - U^n_j}{\Delta t} - \nu \frac{U^{n+1}_{j-1}-2U^{n+1}_j+U^{n+1}_{j+1}}{(\Delta x)^2} = 0 \qquad{} \forall~n\geq 0,\quad{} \forall~ 0\leq j\leq J. \qquad{} (4) $$
>
>For the boundary conditions, we again use equations (3a) or (3b), replacing $n$ by $n+1$.


1) Give the $(J+1)\times(J+1)$ matrices $A^{impl}_1$ and $A^{impl}_2$ such that this scheme reads $A^{impl}_1 U^{n+1}=U^n$ with convention (3a), and $A^{impl}_2 U^{n+1}= U^n$ with convention (3b). Show that both matrices are invertible. *Again, introduce the dimensionless number $\alpha = \frac{\nu \Delta t}{\Delta x^2}$.*


2) **Numerical properties**

>**Remark:** As in the previous part, one can show that the scheme is consistent (replace $n$ by $n+1$ in the definition of the truncation error), with the same orders.

**Stability in the $\ell_{\Delta x}^2$ norm:** Show that scheme (4) with the boundary conditions (3a) is unconditionally stable in the $\ell_{\Delta x}^2$ norm.

>**Remark:** One can then show, as in the previous part, that the scheme is convergent.


3) **Physical properties**

a) **Discrete maximum principle:** Let $i\in\{1,2\}$ and $X,Y\in\mathbb{R}^{J+1}$ such that $A^{impl}_i X = Y$. Show that if $Y\geq 0$ (in the sense that $Y_j\geq 0$ for all $j$), then $X\geq 0$. Deduce that, without any condition on $\alpha$, the scheme satisfies
$$ \forall~n\geq 0,\ \forall~0\leq j\leq J, \quad{} \min_i U^n_i \leq U^{n+1}_j \leq \max_i U^n_i. $$

*Hint: let $e=(1,\dots,1)^T\in\mathbb{R}^{J+1}$, and show that $U^{n+1}-\left(\min_{i} U^n_i\right) e \geq 0$ and $\left(\max_{i}U^n_i\right) e -U^{n+1} \geq 0$.*

b) **Mass conservation:** Depending on the choice between (3a) and (3b), study whether the scheme conserves the mass, that is, whether
$$ \forall n\geq 0, \quad{} \sum_{j=0}^{J} U^{n+1}_j \Delta x = \sum_{j=0}^{J} U^{n}_j\Delta x. $$

*Again, use the vector $e$ and note that the mass reads $e^T\, U^n \Delta x$.*


4) **Implementation**

a) Following what was done for the explicit case, complete the code below to implement schemes (4-3a) and (4-3b).

*As before, define the numerical schemes from the matrices $A^{impl}$ computed in 1), implemented in sparse format. To solve a linear system $AU=b$, use `splu`, the LU factorization for sparse matrices, and its method `solve()`.* **We strongly recommend reading the documentation of these functions.**


In [ ]:
def A_implicite1(alpha, J):
    """
    Compute the matrix A_impl of scheme (4-3a) in sparse format.
    """
    # Your code here.
    pass

def A_implicite2(alpha, J):    
    """
    Compute the matrix A_impl of scheme (4-3b) in sparse format.
    """
    # Your code here.
    pass


In [ ]:
# Test cell to check these matrices.
alpha = 0.1
J = 4
print("A1 = \n", A_implicite1(alpha, J).toarray(), "\n")
print("A2 = \n", A_implicite2(alpha, J).toarray())

In [ ]:
def schema_implicite_chaleur(U0, A, N, dx):
    """
    Compute the iterates U^n of scheme (4-3a) or (4-3b), and return U^N.
    Use splu to factorize A once, and its method solve at each time-step.
    """
    # Your code here.
    pass


b) Repeat the tests done in the explicit case, varying the time-step $\Delta t$ (for instance $\Delta t = 0.1$).

c) Comment on the results and discuss the advantages and drawbacks of the explicit and implicit schemes for this problem.


In [ ]:
# Numerical experiments with the implicit schemes, for several values of dt.

# Your code here.


## Exercise 2: Study of a traveling wave
  
> Let again $\nu>0$ be a diffusion coefficient and $u^0\in C^0(\mathbb{R})$ an initial condition. We now consider a reaction-diffusion equation of the form
>
> $$ 
  \left\{
  \begin{aligned}
  &\displaystyle\frac{\partial u}{\partial t}(t,x)-\nu\frac{\partial^2 u}{\partial x^2}(t,x) = g(u(t,x)), \quad &\forall (t,x) \in\mathbb{R}_+^*\times\mathbb{R}, \\  
  &u(0,x) = u^0(x), \quad &\forall x\in\mathbb{R},
  \end{aligned}
  \right. \qquad{} (5)
$$
>
>where $g:\mathbb{R}\to\mathbb{R}$ is given by
>
>$$ g(u) = ku^2(1-u). \qquad{} (6) $$
>
>This equation, sometimes called the Nagumo equation, describes propagation phenomena. Indeed, for some initial conditions, one can show that the solution approaches a *traveling wave*, that is, a solution of the form
>
> $$ u(t,x) = \beta(x-ct), \qquad{} (7a) $$
>
>where
>
>$$ c= \sqrt{\frac{k\nu}{2}},\qquad{} \beta(\xi) = \frac{e^{-\sqrt{\frac{k}{2\nu}}(\xi-\xi_0)}}{1+e^{-\sqrt{\frac{k}{2\nu}}(\xi-\xi_0)}}, \qquad{} (7b)$$
>
>and $\xi_0\in\mathbb{R}$ depends on the initial condition.
>
>We want to observe this phenomenon numerically, with several schemes. To do so, we work on a bounded domain and consider again $\Omega=]0,L[$ (in practice, $L\gg 1$) and the equation
>
>$$  \left\{
  \begin{aligned}
  &\displaystyle\frac{\partial u}{\partial t}(t,x)-\nu\frac{\partial^2 u}{\partial x^2}(t,x) = g(u(t,x)), \quad &\forall~(t,x) \in\mathbb{R}_+^*\times\Omega,  \\ 
  &\frac{\partial u}{\partial x}(t,0) = 0 = \frac{\partial u}{\partial x}(t,L ), \quad &\forall~t\in\mathbb{R}_+^*, \\ 
  &u(0,x) = u^0(x), \quad &\forall x\in\Omega,
  \end{aligned}
  \right. \qquad{} (8)
$$
>
>still with $g$ given by (6). For the simulations of this exercise, we take
>
>$$ \nu=0.1,\quad{} L=100,\quad{} k=10,\quad{} \text{and}\quad{} u^0(x) =
\left\{
\begin{aligned}
& 1,\quad & x\leq \frac{L}{10}, \\
& 0,\quad & x >\frac{L}{10}.
\end{aligned}
\right.$$
>
>We use the same kind of space and time discretizations as in the previous exercise, with $J=1000$ ($\Delta x = \frac{L}{J}$), and different values of $T$ and $\Delta t$ given below.


### Question 1: Exact solution

Check that the traveling wave $u$ defined in (7) is a solution of the Nagumo equation (5) on $\mathbb{R}$, with $u^0(x) = \beta(x)$.


### Question 2: Fully explicit scheme

>Consider the scheme
>
>$$ \frac{U^{n+1}_j - U^n_j}{\Delta t} - \nu \frac{U^n_{j-1}-2U^n_j+U^n_{j+1}}{(\Delta x)^2} = g(U^n_j) \qquad{} \forall~n\geq 0,\ \forall~ 0\leq j\leq J, $$
>
>with the conventions (3b), which can be rewritten as
>
>$$ U^{n+1} = A^{expl}_2 U^n + \Delta t\, g(U^n), \qquad{} (9) $$
>
>with the notation of Exercise 1, where $g(U^n)$ denotes the vector $\left(g(U^n_j)\right)_{0\leq j\leq J}$.

a) Implement this scheme and plot the solutions obtained at $T=0$, $T=20$, $T=40$, and $T=60$, with $\Delta t = 0.025$ (on the same plot, with different colors).

b) Repeat these experiments with $\Delta t =0.045$. Comment on the results (compute $\alpha = \frac{\nu\Delta t}{\Delta x^2}$ in both cases).


In [ ]:
def g(u, k):
    # Source term.
    # u : unknown
    # k : amplitude of the source term
    return k * u**2 * (1-u)

def u0_2(x, L):
    # Initial condition of Exercise 2.
    # x : position
    # L : length of the domain
    return 1. * (x <= L/10)

In [ ]:
def schema_explicite_reaction_diffusion(U0, A, N, g, k, dt):  
    """
    Compute the iterates U^n of scheme (9), and return U^N.
    """
    # Your code here.
    pass


In [ ]:
# Questions 2.a and 2.b: plot the solutions at T = 0, 20, 40, 60, for dt = 0.025 and dt = 0.045.

# Your code here.


### Question 3: Semi-implicit scheme

>Consider the scheme
>
>$$ \frac{U^{n+1}_j - U^n_j}{\Delta t} - \nu \frac{U^{n+1}_{j-1}-2U^{n+1}_j+U^{n+1}_{j+1}}{(\Delta x)^2} = g(U^n_j) \qquad{} \forall~n\geq 0,\ \forall~ 0\leq j\leq J, $$
>
>with the conventions (3b), which can be rewritten as
>
>$$ A^{impl}_2 U^{n+1} = U^n + \Delta t\, g(U^n). \qquad{} (10) $$

a) Implement this scheme and plot the solutions obtained at $T=0$, $T=20$, $T=40$, and $T=60$, with $\Delta t = 0.045$ (on the same plot, with different colors).

b) Repeat these experiments with $\Delta t = 0.4$. Comment.


In [ ]:
def schema_semi_implicite_reaction_diffusion(U0, A, N, g, k, dt):   
    """
    Compute the iterates U^n of scheme (10), and return U^N.
    """
    # Your code here.
    pass


In [ ]:
# Questions 3.a and 3.b: plot the solutions at T = 0, 20, 40, 60, for dt = 0.045 and dt = 0.4.

# Your code here.


### Question 4: Fully implicit scheme

>Consider the scheme
>
>$$ \frac{U^{n+1}_j - U^n_j}{\Delta t} - \nu \frac{U^{n+1}_{j-1}-2U^{n+1}_j+U^{n+1}_{j+1}}{(\Delta x)^2} = g(U^{n+1}_j) \qquad{} \forall~n\geq 0,\ \forall~0\leq j\leq J, $$
>
>with the conventions (3b), which can be rewritten as
>
>$$ A^{impl}_2 U^{n+1} = U^n + \Delta t\, g(U^{n+1}). \qquad{}(11) $$
>
>Given $U^n$, we must therefore solve a nonlinear system to find $U^{n+1}$. Equivalently, we look for a zero of the function $F^n : \mathbb{R}^{J+1}\to\mathbb{R}^{J+1}$ defined by
>
>$$ F^n (V) = A^{impl}_2 V - \Delta t\, g(V) - U^n. $$
>
>Recall that Newton's method can be used to find an approximate zero:
>
>$$ \left\{\begin{aligned}
&V_0 = U^n, \\
&V_{l+1} = V_l - \left(DF^n(V_l)\right)^{-1} F^n (V_l).
\end{aligned}\right. $$
>
>Once Newton's method has converged, we take the last iterate $V_l$ as $U^{n+1}$.

a) Implement this scheme and plot the solutions obtained at $T=0$, $T=20$, $T=40$, and $T=60$, with $\Delta t = 0.4$ (on the same plot, with different colors).

**Again, use the sparse format, in particular the function `spsolve` to solve sparse linear systems (in Newton's method).**

b) Repeat these experiments with $\Delta t = 0.045$. Comment.


In [ ]:
def dg(u, k):
    # Derivative of the source term.
    # Your code here.
    pass


In [ ]:
def schema_implicite_reaction_diffusion(U0, A, N, g, dg, k, dt, itmax=20, tol=1.e-10):
    """
    Compute the iterates U^n of scheme (11), with Newton's method at each time-step,
    and return U^N.
    """
    # Your code here.
    pass


In [ ]:
# Questions 4.a and 4.b: plot the solutions at T = 0, 20, 40, 60, for dt = 0.4 and dt = 0.045.

# Your code here.
